# 1.5 Data Collection

**Goal:** choose a collection method, recognize common biases, and connect data provenance to reliable AI/ML evaluation.

**Prerequisites:** earlier Module 1 notebooks, especially populations and confounding. **Runtime:** Python 3.10+, standard library only. Run cells in order; saved outputs are included. Every dataset and simulation is synthetic.

## Contents
1. Define the collection plan
2. Surveys
3. Experiments
4. Observational studies
5. Primary and secondary data
6. Data collection, selection, and measurement bias
7. Sampling versus random assignment
8. AI/ML collection and EDA workflow
9. Practice and solutions

## 1. Start with a collection plan

Before collecting rows, define the question, target population, unit, eligibility window, variables, units, and intended decision. Document the sampling frame, selection mechanism, measurement instrument, labeling protocol, and how missingness or failed collection will be recorded.

**Worked example:** for a model that flags damaged parcels from arrival photos, the unit is an arriving parcel, the image is taken before routing decisions, and the label needs an inspection protocol. Collecting photos only of visibly damaged parcels would not support the same question as screening all arriving parcels.

A collection process is part of the measurement system. A dataset can be internally tidy yet systematically omit the cases that matter at deployment.

## 2. Surveys

A survey asks respondents standardized questions to measure reported characteristics, opinions, experiences, or behavior. It can be online, telephone-based, face-to-face, or written. Surveys can target a census or use a sample; an online form is not automatically a probability sample.

A sound survey plan includes clear single-concept questions, a relevant recall window, suitable response choices, neutral wording, an accessible language/mode, and a pilot to catch misunderstandings.

**Worked example:** “How much do you love our fast and reliable service?” is leading and combines speed with reliability. Prefer separate neutral items, such as “How satisfied were you with delivery speed for your most recent order?” with clearly labeled response choices and an option for inapplicability.

Common limitations include frame undercoverage, voluntary-response bias, nonresponse, recall errors, and socially desirable answers. A self-reported behavior may differ from logged behavior. A high response count alone does not prove representativeness.

In [1]:
# Synthetic survey audit: selected sample and respondents are not identical.
invited = {"basic": 80, "premium": 20}
responded = {"basic": 20, "premium": 15}
for group in invited:
    print(group, "response rate:", responded[group]/invited[group])
print("Premium fraction among invited:", invited["premium"]/sum(invited.values()))
print("Premium fraction among respondents:", responded["premium"]/sum(responded.values()))
# Different response propensities can alter composition even after a sound invitation design.

basic response rate: 0.25
premium response rate: 0.75
Premium fraction among invited: 0.2
Premium fraction among respondents: 0.42857142857142855


## 3. Experiments

An experiment deliberately changes a treatment or condition and measures outcomes. In a **randomized experiment**, eligible units are assigned to conditions by a random mechanism. A control group supplies a comparison, and randomization addresses baseline confounding in expectation.

An A/B experiment might assign users to an existing checkout (A) or a new checkout (B), then compare purchase rates over the same fixed window. Define the assignment unit, outcome, exposure rules, and analysis plan before looking for a favorable result.

**Worked example:** randomize by user rather than page view if the same user would otherwise switch designs repeatedly. Persist the assignment and analyze outcomes at an appropriate unit. A small observed difference can arise by chance; point estimates alone do not quantify uncertainty.

Practical threats include attrition after assignment, noncompliance, interference between users, different measurement across groups, and repeatedly checking results without an appropriate stopping plan. Blinding, where feasible, can reduce expectancy or assessment effects. Random assignment supports causal comparisons under assumptions; it does not automatically make volunteers representative of all customers.

In [2]:
import random
import statistics
rng = random.Random(14)
# Fixed random allocation, then simulated binary outcomes.
user_ids = list(range(1000))
rng.shuffle(user_ids)
assignment = {uid: ("A" if i < 500 else "B") for i, uid in enumerate(user_ids)}
results = {"A": [], "B": []}
for uid in range(1000):
    group = assignment[uid]
    true_probability = 0.20 if group == "A" else 0.25
    results[group].append(int(rng.random() < true_probability))
for group in ["A", "B"]:
    print(group, "n:", len(results[group]), "observed conversion:", statistics.mean(results[group]))
difference = statistics.mean(results["B"])-statistics.mean(results["A"])
print("Observed B-A difference (percentage points):", 100*difference)
print("Simulation's underlying probabilities differ by 5 percentage points.")
assert len(results["A"]) == len(results["B"]) == 500

A n: 500 observed conversion: 0.182
B n: 500 observed conversion: 0.246
Observed B-A difference (percentage points): 6.4
Simulation's underlying probabilities differ by 5 percentage points.


## 4. Observational Studies

An observational study records exposures and outcomes without assigning the exposure under study. Examples include analyzing historical transactions, reviewing records, or following a naturally exposed group over time.

Common designs include a cross-sectional snapshot, a cohort followed through time, and a case-control design that starts by selecting outcomes and looks back at exposures. “Retrospective” describes looking back at existing events/records; “prospective” follows data forward from a defined start. These design terms describe different aspects and can overlap.

**Worked example:** comparing churn among users who independently chose monthly versus annual plans is observational. The groups may differ in commitment, tenure, or budget. Their observed churn difference is not automatically the effect of forcing a change in plan.

Observational data can be valuable when experiments are impractical or unethical. Causal claims require explicit assumptions and methods addressing confounding, selection, and measurement. Correlation alone does not supply those assumptions.

| Question | Experiment | Observational study |
|---|---|---|
| Who chooses exposure? | Researcher/design | Naturally occurring process or participants |
| Random assignment possible? | Yes, in randomized experiments | Not for the exposure being observed |
| Main strength | Stronger causal comparison under design assumptions | Real-world coverage and feasibility |
| Main concern | Compliance, attrition, interference, generalization | Confounding, selection, measurement, temporal ambiguity |

## 5. Primary and Secondary Data

**Primary data** are collected directly for the current investigation, such as a purpose-built survey or experiment. **Secondary data** were collected previously or for another purpose, such as an existing log, published dataset, or administrative record.

This is about provenance and purpose, not intrinsic quality. A well-documented secondary dataset can be excellent; a poorly designed primary survey can be biased. One team's primary data can become another team's secondary data. Existing internal logs are often secondary to a new research question.

Check definitions, units, time coverage, inclusion rules, missingness, label construction, permissions for intended use, and changes in collection systems. A historical “customer” field might mean accounts, households, or purchasers depending on the source.

## 6. Data Collection Bias

Data collection bias is a systematic distortion introduced by who enters the dataset or how values are recorded. Distinguish it from random variation: collecting more data under the same biased mechanism can make the wrong answer more precise.

### Selection Bias

Selection bias occurs when inclusion or retention is related to characteristics relevant to the question, distorting the intended comparison or target population. Examples include surveying only active app users about all customers, analyzing only surviving products, losing dissatisfied respondents to follow-up, or collecting photos only in daylight for an all-day detector.

**Coverage bias** concerns gaps between frame and target. **Nonresponse bias** occurs when respondents differ from nonrespondents in relevant ways. **Survivorship bias** is a selection problem that omits failed or discontinued cases. Labels observed only after a selection decision (for example, repayment observed only for approved loans) can also distort ML evaluation.

The simulation below deliberately changes the population composition. The large biased dataset does not become representative merely because it has many rows.

In [3]:
# Target: equal numbers of two groups; outcome is 1 for group H, 0 for group L.
population = [1]*5000 + [0]*5000
# Selection mechanism retains all H but just 10% of L.
selected = [1]*5000 + [0]*500
rng = random.Random(23)
random_sample = rng.sample(population, 400)
print("Target population mean:", statistics.mean(population))
print("Large selected dataset: n=", len(selected), "mean=", statistics.mean(selected))
print("Probability sample: n=", len(random_sample), "mean=", statistics.mean(random_sample))
print("Large selected-set discrepancy:", statistics.mean(selected)-statistics.mean(population))
assert statistics.mean(selected) > 0.90

Target population mean: 0.5
Large selected dataset: n= 5500 mean= 0.9090909090909091
Probability sample: n= 400 mean= 0.5075
Large selected-set discrepancy: 0.40909090909090906


### Measurement Bias

Measurement bias is systematic error in recorded values relative to the intended construct or reference. Examples include a sensor that reads 2 units too high, leading survey wording, inconsistent label definitions, or a classifier's labels being assessed differently across groups.

A measurement can have **random noise** (variation without a systematic shift under a stated model) and **systematic error** at the same time. Repeated measurements can reduce some random noise but do not remove a calibration offset. High repeatability (**reliability**) does not guarantee that the instrument measures the intended construct (**validity**).

**Worked example:** an uncalibrated scale reports every mass 2 kg too high. Every reading can be consistent while the mean is wrong by 2 kg. For text labels, a written rubric, annotator training, agreement review, and adjudication can reduce inconsistent interpretation, but high agreement alone does not prove labels are valid.

### Processing and missing-data problems

Unit conversions, duplicate records, broken joins, changed logging schemas, and sentinel values such as `-999` can create further distortions. Retain missingness explicitly and investigate its mechanism before imputing. Missing values often reflect the collection process rather than random omissions.

In [4]:
true_masses = [50, 55, 60, 65, 70]
measured_masses = [x+2 for x in true_masses]
print("True mean:", statistics.mean(true_masses))
print("Uncalibrated mean:", statistics.mean(measured_masses))
print("Systematic shift:", statistics.mean(measured_masses)-statistics.mean(true_masses))
print("Spread unchanged:", statistics.pstdev(true_masses), statistics.pstdev(measured_masses))
# Correction assumes the +2 offset was established using an external reference.
corrected = [x-2 for x in measured_masses]
assert corrected == true_masses

True mean: 60
Uncalibrated mean: 62
Systematic shift: 2
Spread unchanged: 7.0710678118654755 7.0710678118654755


## 7. Random sampling is different from random assignment

**Random sampling** selects units from a frame and supports population generalization under the sampling design. **Random assignment** allocates selected units to treatments and supports causal comparison under experimental assumptions. A study may use either, both, or neither.

**Worked example:** a randomized A/B test among self-selected beta volunteers randomizes the interface but does not randomly sample all users. Its causal comparison among participants can be more credible than its generalization to nonparticipants. A random survey of the user base can generalize descriptive preferences while not establishing the effect of a chosen plan on churn.

| Process | Primary question it addresses | Does not automatically solve |
|---|---|---|
| Random sampling | Who should the observed data represent? | Confounding of naturally chosen exposures |
| Random assignment | Can group outcome differences be attributed to treatment? | Undercoverage of the target population |

## 8. AI/ML workflow: from collection to EDA

For a customer-support routing model:

1. Define the prediction moment: when the ticket first arrives.
2. Define the unit: one ticket, with customer IDs retained for grouping.
3. Collect incoming text and metadata from all supported channels and relevant periods.
4. Define the label: correct routing category under a documented rubric, including how ambiguous cases are handled.
5. Audit missing channels, duplicate tickets, language coverage, and labeling differences.
6. Separate training and evaluation data using a design appropriate to time and customer overlap. Exclude outcome-derived fields and fit preprocessing on training data only.
7. Perform EDA: field types, missingness, frequencies, ranges, label balance, and subgroup coverage. Investigate unexpected values before choosing transformations.
8. Document intended deployment scope and remaining gaps; update collection if important conditions are absent.

The following quality audit identifies issues without silently fixing them. A duplicate ID may be an error or a versioned event; a missing duration may be unavailable rather than zero. Resolution depends on the data contract and intended use.

In [5]:
from collections import Counter
records = [
    {"id": "T1", "channel": "web", "language": "en", "duration": 5.0, "label": "billing"},
    {"id": "T2", "channel": "web", "language": "en", "duration": None, "label": "technical"},
    {"id": "T2", "channel": "web", "language": "en", "duration": None, "label": "technical"},
    {"id": "T3", "channel": "phone", "language": "hi", "duration": -4.0, "label": None},
]
id_counts = Counter(r["id"] for r in records)
print("Repeated IDs:", {key: n for key, n in id_counts.items() if n > 1})
print("Channel counts:", dict(Counter(r["channel"] for r in records)))
print("Language counts:", dict(Counter(r["language"] for r in records)))
print("Missing labels:", sum(r["label"] is None for r in records))
print("Missing durations:", sum(r["duration"] is None for r in records))
print("Invalid negative durations:", [r["id"] for r in records
      if r["duration"] is not None and r["duration"] < 0])
print("Duration is a quality-audit field here; final duration is unavailable at ticket arrival.")
assert id_counts["T2"] == 2
assert sum(r["label"] is None for r in records) == 1

Repeated IDs: {'T2': 2}
Channel counts: {'web': 3, 'phone': 1}
Language counts: {'en': 3, 'hi': 1}
Missing labels: 1
Missing durations: 2
Invalid negative durations: ['T3']
Duration is a quality-audit field here; final duration is unavailable at ticket arrival.


## Practice

1. You want satisfaction among all customers but survey only customers who open a promotional email. Identify two possible selection problems.
2. Is analysis of old transaction logs primary or secondary data for a new fraud study?
3. A randomized checkout experiment enrolls only volunteers. Which issue does assignment address, and which does it leave open?
4. A thermometer reads 1.5 degrees too high every time. Is this mainly random noise or systematic measurement error? Will more readings fix the offset?
5. Compare an observational study of exercise habits with a randomized exercise intervention.
6. A form asks “How helpful and fast was our excellent support?” Give a better approach.
7. Why should missing response times not automatically become zero?
8. A model performs well on clear daytime photos. What collection change would you consider before deploying at night?
9. In the quality-audit dataset, is it safe to drop repeated IDs immediately? Why or why not?

## Solutions

1. Frame/coverage restriction to email recipients and openers, plus voluntary response/nonresponse among those invited. Both can depend on engagement or satisfaction.
2. Secondary to the current study because the logs were collected previously for another purpose. Check whether logging and label definitions support the fraud question.
3. Random assignment addresses treatment-group confounding in expectation among participants. Volunteer selection leaves generalization to the full customer base uncertain.
4. Systematic measurement error. Repetition does not remove a fixed offset; calibration against a reference is needed.
5. The first records naturally chosen habits and is vulnerable to confounding such as baseline health. The second assigns the intervention, making causal interpretation more credible under compliance, measurement, and other design assumptions.
6. Separate helpfulness from speed, remove praise/leading wording, specify the relevant interaction, and offer balanced labeled response choices plus an inapplicable option.
7. Missing means unknown or unrecorded, whereas zero means no elapsed time. Substituting zero can bias summaries and trained models.
8. Collect and label representative nighttime and low-light examples, audit relevant devices/conditions, and evaluate on a separate dataset reflecting intended deployment.
9. No. Check whether an ID denotes a unique ticket, an update, or a versioned event. Blind deletion can lose valid history; retaining exact duplicates can overweight cases.

## Module 1 recap and readiness check

You should now be able to define a target population and unit; distinguish parameters from sample statistics; classify data types and measurement scales; identify predictors, targets, and plausible confounders; and audit how collection affects an ML dataset.

Before training a model, write down **what one row means**, **who is missing**, **when each feature becomes available**, and **which population the evaluation is intended to represent**.

**Next:** Module 2 — Descriptive Statistics in the [main syllabus](../README.md).